In [1]:
import os
import numpy as np
import community as community
import random
from tqdm import tqdm, trange
import pickle
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn import metrics


In [2]:
def ale_1d(model, X, feature, edges, class_index=0):
    """
    Compute 1D Accumulated Local Effects (ALE) for a single feature on
    predicted class probability (classification). Uses fixed bin edges.

    Parameters
    ----------
    model : fitted classifier with predict_proba
    X : pandas.DataFrame (data used to estimate the ALE; e.g., test or train)
    feature : str
    edges : np.array of shape (n_bins+1,)
    class_index : int (which class's probability to explain)

    Returns
    -------
    pandas.DataFrame with columns:
      - 'x': bin midpoints
      - 'ale': centered accumulated local effect per bin
      - 'count': number of samples in each bin
    """
    X = X.copy()
    n_bins = len(edges) - 1
    local_effects = []
    counts = []

    for k in range(n_bins):
        lower, upper = edges[k], edges[k+1]
        # last bin inclusive on upper bound to capture max
        if k == n_bins - 1:
            mask = (X[feature] >= lower) & (X[feature] <= upper)
        else:
            mask = (X[feature] >= lower) & (X[feature] < upper)

        n_k = int(mask.sum())
        counts.append(n_k)

        if n_k == 0:
            local_effects.append(0.0)
            continue

        # Create perturbed datasets
        X_upper = X.loc[mask].copy()
        X_lower = X.loc[mask].copy()
        X_upper[feature] = upper
        X_lower[feature] = lower

        # Class probability at bin edges
        p_upper = model.predict_proba(X_upper)[:, class_index]
        p_lower = model.predict_proba(X_lower)[:, class_index]

        # Average local effect in this bin
        local_effects.append((p_upper - p_lower).mean())

    # Accumulate local effects to get ALE curve
    ale = np.cumsum(local_effects)
    midpoints = (edges[:-1] + edges[1:]) / 2.0

    # Center the ALE curve (weighted by bin counts, standard in ALE)
    weights = np.array(counts, dtype=float)
    wsum = weights.sum()
    mean_ale = (ale * weights).sum() / (wsum if wsum > 0 else 1.0)
    ale_centered = ale - mean_ale

    return pd.DataFrame({"x": midpoints, "ale": ale_centered, "count": counts})


In [3]:
RUNS = 10

def node_dataset_gen(X, entropy_values, us):
    kmeans_seed = random.randint(0, 10000)
    kmeans = KMeans(n_clusters=2, random_state=kmeans_seed).fit(entropy_values.reshape(-1, 1))
    cutoff = np.mean(kmeans.cluster_centers_)
    y = np.where(entropy_values < cutoff, 0, 1)
    if us != None:
        stab_unstab = np.bincount(y)
        num_unstab = stab_unstab[1]
        num_stab = int(num_unstab / us)
        lowest_entropy_indices = list(entropy_values.argsort()[:num_stab][::-1])
        highest_entropy_indices = list(entropy_values.argsort()[-num_unstab:][::-1])
        X_lowest = X.iloc[lowest_entropy_indices]
        X_highest = X.iloc[highest_entropy_indices]
        X = pd.concat([X_lowest, X_highest])
        y = [0 for _ in range(num_stab)] + [1 for _ in range(num_unstab)]
        y = pd.DataFrame(y, index=X.index, columns=['Stability'])
    else:
        y = pd.DataFrame(y, index=X.index, columns=['Stability'])
    split_seed = random.randint(0, 10000)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=split_seed)
    return X_train, X_test, y_train, y_test, cutoff


def pair_dataset_gen(X, entropy_values):
    lowest_entropy_indices = list(entropy_values.argsort()[:500][::-1])
    highest_entropy_indices = list(entropy_values.argsort()[-500:][::-1])
    X_lowest = X.iloc[lowest_entropy_indices]
    X_highest = X.iloc[highest_entropy_indices]
    X = pd.concat([X_lowest, X_highest])
    y = [0 for _ in range(500)] + [1 for _ in range(500)]
    y = pd.DataFrame(y, index=X.index, columns=['Same Community'])
    split_seed = random.randint(0, 10000)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=split_seed)
    return X_train, X_test, y_train, y_test


def save_dataset(X_train, X_test, y_train, y_test):
    X_train.to_csv(os.path.join(results_folder, 'X_train.csv'))
    y_train.to_csv(os.path.join(results_folder, 'y_train.csv'))
    X_test.to_csv(os.path.join(results_folder, 'X_test.csv'))
    y_test.to_csv(os.path.join(results_folder, 'y_test.csv'))
    return


def create_and_save_data(feats_fil, entropies_fil, results_folder, us, mode):
    X = pd.read_csv(feats_fil, index_col=0)
    entrops = pd.read_csv(entropies_fil, index_col=0)
    entropy_values = np.array(entrops['Entropy'])
    entropy_values = entropy_values.reshape(-1)
    if mode == 'node':
        X_train, X_test, y_train, y_test, cutoff = node_dataset_gen(X, entropy_values, us)
    elif mode == 'pair':
        X_train, X_test, y_train, y_test = pair_dataset_gen(X, entropy_values)
    save_dataset(X_train, X_test, y_train, y_test)
    if mode == 'node':
        label_counts = np.bincount(y_train['Stability'])
        results_dict = {'Stable Nodes': label_counts[0], 'Unstable Nodes': label_counts[1],
                        'Stability Cutoff': cutoff, 'Undersampling Level': us}
    elif mode == 'pair':
        label_counts = np.bincount(y_train['Same Community'])
        results_dict = {'Different Communities': label_counts[0], 'Same Communities': label_counts[1],
                        'Undersampling Level': us}
    return X_train, y_train, X_test, y_test, results_dict



def train(X_train, X_test, y_train, y_test, n_splits=5, n_bins=20):
    """
    Train RandomForest with Stratified CV and compute 1D ALE curves per feature.

    Returns
    -------
    ale_curves : dict[str, pandas.DataFrame]
        For each feature, a DataFrame with columns:
            - 'x' (bin midpoints),
            - 'ale_mean',
            - 'ale_std',
            - 'avg_bin_count'
    accuracy_scores : list[float]
    balanced_accuracy_scores : list[float]
    """
    feature_list = list(X_train.columns)
    data = np.array(X_train)
    labels = np.squeeze(np.array(y_train))
    accuracy_scores = []
    balanced_accuracy_scores = []

    # Precompute fixed bin edges from X_test quantiles (stable across folds)
    quantiles = np.linspace(0, 1, n_bins + 1)
    feature_edges = {}
    for f in feature_list:
        edges = np.quantile(X_test[f], quantiles)
        # Ensure strictly increasing edges; fallback to linspace if too many duplicates
        edges_unique = np.unique(edges)
        if len(edges_unique) < 2:
            mn = float(X_test[f].min())
            mx = float(X_test[f].max())
            edges_unique = np.linspace(mn, mx, n_bins + 1)
        feature_edges[f] = edges_unique

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    rf = RandomForestClassifier(random_state=42)

    # Collect+ ALE curves per feature across folds/runs
    ale_stack = {f: [] for f in feature_list}

    fold_count = 0
    for run in trange(1, RUNS + 1):
        for train_idx, val_idx in skf.split(data, labels):
            fold_count += 1

            X_train_fold = X_train.iloc[train_idx]
            X_val_fold   = X_train.iloc[val_idx]
            if hasattr(y_train, 'iloc'):
                y_train_fold = y_train.iloc[train_idx]
                y_val_fold   = y_train.iloc[val_idx]
            else:
                y_train_fold = y_train[train_idx]
                y_val_fold   = y_train[val_idx]

            model = rf.fit(X_train_fold, y_train_fold)
            predictions = rf.predict(X_val_fold)

            accuracy_scores.append(metrics.accuracy_score(y_val_fold, predictions))
            balanced_accuracy_scores.append(metrics.balanced_accuracy_score(y_val_fold, predictions))

            # Choose class to explain (binary: positive class by label order)
            classes = list(model.classes_)
            if len(classes) == 2:
                pos_class = max(classes)  # convention: positive is larger label
                class_index = classes.index(pos_class)
            else:
                class_index = 0  # explain first class; can loop over all if you wish

            # Compute ALE on X_test for each feature using the fixed edges
            for f in feature_list:
                df_ale = ale_1d(model, X_test, f, feature_edges[f], class_index=class_index)
                df_ale['run'] = fold_count
                ale_stack[f].append(df_ale)

    # Aggregate ALE curves: average and std across folds/runs
    ale_curves = {}
    for f in feature_list:
        df_all = pd.concat(ale_stack[f], ignore_index=True)
        grp = df_all.groupby('x', sort=True)
        mean_ale  = grp['ale'].mean()
        std_ale   = grp['ale'].std()
        mean_cnt  = grp['count'].mean()
        ale_curves[f] = pd.DataFrame({
            'x': mean_ale.index.values,
            'ale_mean': mean_ale.values,
            'ale_std': std_ale.values,
            'avg_bin_count': mean_cnt.values
        })

    return ale_curves, accuracy_scores, balanced_accuracy_scores

In [8]:
mu = 'mu_0_2'
algorithm = 'Infomap'
feats_folder = f'LFR_Graph_Data/Community_Data/{algorithm}/Node_Features/'
entropies_folder = f'LFR_Graph_Data/Community_Data/{algorithm}/Node_Entropies/'
results_folder = f'LFR_Graph_Data/Community_Data/{algorithm}/results/{mu}'

us, mode = 0.75, 'node'

# Create results directory if it doesn't exist
if not os.path.exists(results_folder):
    os.makedirs(results_folder)

# Dictionary to store all results
all_results = {}

for feats_fil in os.listdir(feats_folder):
    if feats_fil.endswith(f'{mu}_features.csv'):
        # Extract graph name (e.g., 'graph_01_mu_0_3')
        graph_name = feats_fil.replace('_features.csv', '')
        
        # Construct corresponding entropy filename
        entropies_csv = feats_fil.replace('_features.csv', '_entropies.csv')
        
        # Full file paths
        feats_path = os.path.join(feats_folder, feats_fil)
        entropies_path = os.path.join(entropies_folder, entropies_csv)

        individual_results_path = os.path.join(results_folder, f"{graph_name}_results.pkl")

        if os.path.exists(individual_results_path):
            with open(individual_results_path, 'rb') as fp:
                graph_results = pickle.load(fp)
            all_results[graph_name] = graph_results
            print(f"Skipping {individual_results_path} already exists.")
            continue
                

        print(f"Processing {graph_name}...")
        
        # Skip if entropy file doesn't exist
        if not os.path.exists(entropies_path):
            print(f"Warning: Entropy file {entropies_csv} not found. Skipping {graph_name}")
            continue
        
        # Create and train
        X_train, y_train, X_test, y_test, results_dict = create_and_save_data(
            feats_path, entropies_path, results_folder, us, mode
        )
        
        # Train model and get ALE
        ale_curves, accuracy_scores, balanced_accuracy_scores = train(
            X_train, X_test, y_train, y_test
        )
        
        # Store results in nested structure
        graph_results = {
            'ale_curves': ale_curves,  # Dictionary with feature->DataFrame
            'accuracy_scores': accuracy_scores,
            'balanced_accuracy_scores': balanced_accuracy_scores,
            'train_test_info': {
                'X_train_shape': X_train.shape,
                'X_test_shape': X_test.shape,
                'y_train_distribution': y_train['Stability'].value_counts().to_dict(),
                'y_test_distribution': y_test['Stability'].value_counts().to_dict()
            }
        }
        
        # Add original results_dict entries
        graph_results.update(results_dict)
        
        # Save to all_results dictionary
        all_results[graph_name] = graph_results
        
        # Also save individual graph results to separate file
        individual_results_path = os.path.join(results_folder, f"{graph_name}_results.pkl")
        with open(individual_results_path, 'wb') as fp:
            pickle.dump(graph_results, fp)
        
        print(f"Completed {graph_name}")

# Save aggregated results 
aggregated_results_path = os.path.join(results_folder, 'all_graphs_results.pkl')
with open(aggregated_results_path, 'wb') as fp:
    pickle.dump(all_results, fp)

print(f"Processed {len(all_results)} graphs")
print(f"Results saved to {aggregated_results_path}")

Processing graph_01_mu_0_2...


/var/data/python/lib/python3.13/site-packages/sklearn/base.py:1336: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (2). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


ValueError: Shape of passed values is (2333, 1), indices imply (2000, 1)